In [1]:
!rm -rf diploma_centpy_parallelization_py
# Флаг -b указывает конкретную ветку
!git clone -b feature/jax-centpy https://github.com/filkinc/diploma_centpy_parallelization_py.git
%cd diploma_centpy_parallelization_py
%cd /content/diploma_centpy_parallelization_py/jax_centpy

# Установка зависимостей
!pip install centpy pandas matplotlib seaborn

Cloning into 'diploma_centpy_parallelization_py'...
remote: Enumerating objects: 263, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 263 (delta 6), reused 19 (delta 5), pack-reused 215 (from 1)
Receiving objects: 100% (263/263), 22.26 MiB | 17.16 MiB/s, done.
Resolving deltas: 100% (100/100), done.
/content/diploma_centpy_parallelization_py
/content/diploma_centpy_parallelization_py/jax_centpy


In [2]:
import os
import time
import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import numpy as np
from google.colab import files

from core import Pars2d, Equation2d, Pars1d, Equation1d
from solver import Solver1d, FastSolver1d
from boundaries import periodic_bc_2d, neumann_bc_2d, dirichlet_riemann_bc_2d
from equations import make_euler_riemann_2d, make_euler_isentropic_vortex_2d
from schemes import compute_rhs_sd2_2d
from limiters import monotonized_central, minmod
from richardson import self_convergence_analysis

In [4]:
def exact_riemann_solver(rho_L, u_L, p_L, rho_R, u_R, p_R, x_arr, t, x0=0.5, gamma=1.4):
    """
    Точный решатель Римана для уравнений Эйлера (идеальный газ).
    Адаптировано по книге E.F. Toro "Riemann Solvers and Numerical Methods for Fluid Dynamics".
    """
    if t == 0:
        rho = np.where(x_arr < x0, rho_L, rho_R)
        u = np.where(x_arr < x0, u_L, u_R)
        p = np.where(x_arr < x0, p_L, p_R)
        return rho, u, p

    # Вычисление скоростей звука
    c_L = np.sqrt(gamma * p_L / rho_L)
    c_R = np.sqrt(gamma * p_R / rho_R)

    # 1. Итерационный процесс Ньютона-Рафсона для нахождения давления p_star в звездной области
    def f(p, rho_K, p_K, c_K):
        if p > p_K: # Ударная волна
            A = 2.0 / ((gamma + 1.0) * rho_K)
            B = (gamma - 1.0) / (gamma + 1.0) * p_K
            return (p - p_K) * np.sqrt(A / (p + B))
        else:       # Волна разрежения
            return 2.0 * c_K / (gamma - 1.0) * ((p / p_K)**((gamma - 1.0) / (2.0 * gamma)) - 1.0)

    def dfdp(p, rho_K, p_K, c_K):
        if p > p_K:
            A = 2.0 / ((gamma + 1.0) * rho_K)
            B = (gamma - 1.0) / (gamma + 1.0) * p_K
            return np.sqrt(A / (p + B)) * (1.0 - (p - p_K) / (2.0 * (p + B)))
        else:
            return (1.0 / (rho_K * c_K)) * (p / p_K)**(-(gamma + 1.0) / (2.0 * gamma))

    p_old = 0.5 * (p_L + p_R) # Начальное приближение
    for _ in range(100):
        f_L = f(p_old, rho_L, p_L, c_L)
        f_R = f(p_old, rho_R, p_R, c_R)
        df_L = dfdp(p_old, rho_L, p_L, c_L)
        df_R = dfdp(p_old, rho_R, p_R, c_R)
        p_new = p_old - (f_L + f_R + u_R - u_L) / (df_L + df_R)
        if abs(p_new - p_old) < 1e-6:
            break
        p_old = p_new
    p_star = abs(p_old)

    # 2. Вычисление скорости u_star
    u_star = 0.5 * (u_L + u_R + f(p_star, rho_R, p_R, c_R) - f(p_star, rho_L, p_L, c_L))

    # 3. Сэмплирование решения для каждой точки x
    rho_exact = np.zeros_like(x_arr)
    u_exact = np.zeros_like(x_arr)
    p_exact = np.zeros_like(x_arr)

    S = (x_arr - x0) / t

    for i, s in enumerate(S):
        if s < u_star:  # Левая волна
            if p_star > p_L: # Левая ударная волна
                S_L = u_L - c_L * np.sqrt((gamma + 1.0)/(2.0*gamma) * (p_star/p_L) + (gamma - 1.0)/(2.0*gamma))
                if s < S_L:
                    rho_exact[i], u_exact[i], p_exact[i] = rho_L, u_L, p_L
                else:
                    rho_star_L = rho_L * (p_star / p_L + (gamma-1.0)/(gamma+1.0)) / ((gamma-1.0)/(gamma+1.0) * (p_star/p_L) + 1.0)
                    rho_exact[i], u_exact[i], p_exact[i] = rho_star_L, u_star, p_star
            else: # Левая волна разрежения
                S_HL = u_L - c_L
                c_star_L = c_L * (p_star / p_L)**((gamma - 1.0) / (2.0 * gamma))
                S_TL = u_star - c_star_L
                if s < S_HL:
                    rho_exact[i], u_exact[i], p_exact[i] = rho_L, u_L, p_L
                elif s > S_TL:
                    rho_star_L = rho_L * (p_star / p_L)**(1.0 / gamma)
                    rho_exact[i], u_exact[i], p_exact[i] = rho_star_L, u_star, p_star
                else:
                    u_fan = 2.0 / (gamma + 1.0) * (c_L + (gamma - 1.0) / 2.0 * u_L + s)
                    c_fan = 2.0 / (gamma + 1.0) * (c_L + (gamma - 1.0) / 2.0 * (u_L - s))
                    rho_fan = rho_L * (c_fan / c_L)**(2.0 / (gamma - 1.0))
                    p_fan = p_L * (c_fan / c_L)**(2.0 * gamma / (gamma - 1.0))
                    rho_exact[i], u_exact[i], p_exact[i] = rho_fan, u_fan, p_fan
        else: # Правая волна
            if p_star > p_R: # Правая ударная волна
                S_R = u_R + c_R * np.sqrt((gamma + 1.0)/(2.0*gamma) * (p_star/p_R) + (gamma - 1.0)/(2.0*gamma))
                if s > S_R:
                    rho_exact[i], u_exact[i], p_exact[i] = rho_R, u_R, p_R
                else:
                    rho_star_R = rho_R * (p_star / p_R + (gamma-1.0)/(gamma+1.0)) / ((gamma-1.0)/(gamma+1.0) * (p_star/p_R) + 1.0)
                    rho_exact[i], u_exact[i], p_exact[i] = rho_star_R, u_star, p_star
            else: # Правая волна разрежения
                S_HR = u_R + c_R
                c_star_R = c_R * (p_star / p_R)**((gamma - 1.0) / (2.0 * gamma))
                S_TR = u_star + c_star_R
                if s > S_HR:
                    rho_exact[i], u_exact[i], p_exact[i] = rho_R, u_R, p_R
                elif s < S_TR:
                    rho_star_R = rho_R * (p_star / p_R)**(1.0 / gamma)
                    rho_exact[i], u_exact[i], p_exact[i] = rho_star_R, u_star, p_star
                else:
                    u_fan = 2.0 / (gamma + 1.0) * (-c_R + (gamma - 1.0) / 2.0 * u_R + s)
                    c_fan = 2.0 / (gamma + 1.0) * (c_R - (gamma - 1.0) / 2.0 * (u_R - s))
                    rho_fan = rho_R * (c_fan / c_R)**(2.0 / (gamma - 1.0))
                    p_fan = p_R * (c_fan / c_R)**(2.0 * gamma / (gamma - 1.0))
                    rho_exact[i], u_exact[i], p_exact[i] = rho_fan, u_fan, p_fan

    return rho_exact, u_exact, p_exact

In [17]:
import jax.numpy as jnp
from typing import Callable, NamedTuple
from core import Equation1d

def make_euler_riemann_1d(test_case: str = "Sod", gamma: float = 1.4) -> Equation1d:
    """
    Генератор уравнений Эйлера 1D для различных постановок задачи Римана.
    Исправлен баг с broadcasting в spectral_radius.
    """
    def compute_pressure(q):
        rho = q[..., 0]
        u = q[..., 1] / rho
        E = q[..., 2]
        return (gamma - 1.0) * (E - 0.5 * rho * u**2)

    def flux(q):
        rho = q[..., 0]
        rhou = q[..., 1]
        E = q[..., 2]
        rhosafe = jnp.maximum(rho, 1e-10)
        u = rhou / rhosafe
        p = compute_pressure(q)
        return jnp.stack([rhou, rhou * u + p, u * (E + p)], axis=-1)

    def spectral_radius(q):
        rho = q[..., 0]
        rhou = q[..., 1]
        rhosafe = jnp.maximum(rho, 1e-10)
        u = rhou / rhosafe
        p = jnp.maximum(compute_pressure(q), 1e-10)
        c = jnp.sqrt(gamma * p / rhosafe)

        # ИСПРАВЛЕНИЕ: Добавляем [..., None], чтобы размерность стала (N, 1) вместо (N,)
        # Это позволит корректно выполнить a * (u_plus - u_minus) в schemes.py
        a = jnp.abs(u) + c
        return a[..., None]

    def initial_data(x):
        if test_case == "Sod":
            rho_L, u_L, p_L = 1.0, 0.0, 1.0
            rho_R, u_R, p_R = 0.125, 0.0, 0.1
        elif test_case == "Lax":
            rho_L, u_L, p_L = 0.445, 0.698, 3.528
            rho_R, u_R, p_R = 0.5, 0.0, 0.571
        elif test_case == "Toro123":
            rho_L, u_L, p_L = 1.0, -2.0, 0.4
            rho_R, u_R, p_R = 1.0, 2.0, 0.4
        else:
            raise ValueError(f"Неизвестный тест: {test_case}")

        condition = x < 0.5
        rho = jnp.where(condition, rho_L, rho_R)
        u = jnp.where(condition, u_L, u_R)
        p = jnp.where(condition, p_L, p_R)

        E = p / (gamma - 1.0) + 0.5 * rho * u**2
        return jnp.stack([rho, rho * u, E], axis=-1)

    def neumann_bc(u, nghost):
        return jnp.pad(u, ((nghost, nghost), (0, 0)), mode='edge')

    return Equation1d(
        flux=flux,
        spectral_radius=spectral_radius,
        initial_data=initial_data,
        boundary_handler=neumann_bc,
        name=f"Euler 1D - {test_case}"
    )

In [ ]:
eqn = make_euler_riemann_1d(test_case="Lax")
pars = Pars1d(x_init=0.0, x_final=1.0, t_final=0.14, dt_out=0.005, J=200, cfl=0.45, scheme="sd2")
solver = Solver1d(pars, eqn, schemename='sd2', limitername='minmod')

In [29]:
def run_and_animate_exact_vs_num(test_case="Sod", J=200, limiter='minmod'):
    test_params = {
        "Sod": {"t_final": 0.2, "L": (1.0, 0.0, 1.0), "R": (0.125, 0.0, 0.1)},
        "Lax": {"t_final": 0.14, "L": (0.445, 0.698, 3.528), "R": (0.5, 0.0, 0.571)},
        "Toro123": {"t_final": 0.15, "L": (1.0, -2.0, 0.4), "R": (1.0, 2.0, 0.4)}
    }

    t_final = test_params[test_case]["t_final"]
    W_L = test_params[test_case]["L"]
    W_R = test_params[test_case]["R"]

    pars = Pars1d(x_init=0.0, x_final=1.0, t_final=t_final, dt_out=0.005, J=J, cfl=0.45, scheme='sd2')
    eqn = make_euler_riemann_1d(test_case=test_case)
    solver = Solver1d(pars, eqn, scheme_name='sd2', limiter_name=limiter)

    print(f"Рассчитываем численное решение для {test_case} (сетка: {J}, лимитер: {limiter})...")

    # Достаем массивы с правильным ключом 'un'
    solution = solver.solve()
    x_num = np.array(solution["x"])
    t_num = np.array(solution["t"])
    u_num = np.array(solution["u_n"])  # <-- ИСПРАВЛЕНИЕ ЗДЕСЬ

    fig, axes = plt.subplots(3, 1, figsize=(8, 10))
    fig.suptitle(f"{test_case} Test | Exact vs {limiter} (J={J})", fontsize=14)

    line_rho_num, = axes[0].plot([], [], 'ro', markersize=3, label='Numerical (SD2)')
    line_rho_ex, = axes[0].plot([], [], 'k-', lw=2, label='Exact')
    axes[0].set_ylabel('Density', fontsize=12)
    axes[0].legend()
    axes[0].grid(True)

    line_u_num, = axes[1].plot([], [], 'ro', markersize=3)
    line_u_ex, = axes[1].plot([], [], 'k-', lw=2)
    axes[1].set_ylabel('Velocity', fontsize=12)
    axes[1].grid(True)

    line_p_num, = axes[2].plot([], [], 'ro', markersize=3)
    line_p_ex, = axes[2].plot([], [], 'k-', lw=2)
    axes[2].set_ylabel('Pressure', fontsize=12)
    axes[2].set_xlabel('x', fontsize=12)
    axes[2].grid(True)

    # Настраиваем оси по точному решению в момент времени t_final
    rho_ex_final, u_ex_final, p_ex_final = exact_riemann_solver(*W_L, *W_R, x_num, t_final)

    axes[0].set_xlim(0, 1); axes[0].set_ylim(min(W_R[0], W_L[0], rho_ex_final.min()) - 0.1, max(W_R[0], W_L[0], rho_ex_final.max()) + 0.1)
    axes[1].set_xlim(0, 1); axes[1].set_ylim(min(W_R[1], W_L[1], u_ex_final.min()) - 0.1, max(W_R[1], W_L[1], u_ex_final.max()) + 0.1)
    axes[2].set_xlim(0, 1); axes[2].set_ylim(min(W_R[2], W_L[2], p_ex_final.min()) - 0.1, max(W_R[2], W_L[2], p_ex_final.max()) + 0.1)

    plt.tight_layout()

    x_exact = np.linspace(0, 1, 1000)

    def animate(i):
        data = u_num[i]
        rho = data[:, 0]
        vel = data[:, 1] / rho
        p = (1.4 - 1.0) * (data[:, 2] - 0.5 * rho * vel**2)

        line_rho_num.set_data(x_num, rho)
        line_u_num.set_data(x_num, vel)
        line_p_num.set_data(x_num, p)

        curr_t = t_num[i]

        # Обход деления на 0 при первом кадре
        if curr_t < 1e-10:
            rho_ex = np.where(x_exact < 0.5, W_L[0], W_R[0])
            u_ex = np.where(x_exact < 0.5, W_L[1], W_R[1])
            p_ex = np.where(x_exact < 0.5, W_L[2], W_R[2])
        else:
            rho_ex, u_ex, p_ex = exact_riemann_solver(*W_L, *W_R, x_exact, curr_t)

        line_rho_ex.set_data(x_exact, rho_ex)
        line_u_ex.set_data(x_exact, u_ex)
        line_p_ex.set_data(x_exact, p_ex)

        axes[0].set_title(f'Time: {curr_t:.3f}', fontsize=12)
        return line_rho_num, line_u_num, line_p_num, line_rho_ex, line_u_ex, line_p_ex

    anim = animation.FuncAnimation(fig, animate, frames=len(t_num), interval=50, blit=False)

    # Сохраняем в MP4
    video_filename = f"{test_case}_{limiter}_J{J}.mp4"
    anim.save(video_filename, writer='ffmpeg', fps=15)
    plt.close(fig)

    try:
        from google.colab import files
        files.download(video_filename)
    except Exception:
        pass

    return HTML(anim.to_html5_video())

In [30]:
# Запуск классической задачи Сода с лимитером minmod
anim = run_and_animate_exact_vs_num(test_case="Sod", J=200, limiter="minmod")
anim

Рассчитываем численное решение для Sod (сетка: 200, лимитер: minmod)...
Starting simulation: Euler 1D - Sod
Grid: 200 points, Scheme: SD2/minmod
Simulation finished in 1.2006s
Total steps: 199


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>